## CHANGELOG.md

# التعديلات اللي اتعملت، ملف ملف (مكان كل تعديل بالظبط)

## 1. `build_dataset_from_medquad.py`
- **مكان التعديل:** دالة `build_document()`، السطر اللي فيه `key = answer[:200]`.
- **التعديل:** بقى فيه دالة جديدة `_dedup_key()` بتعمل normalize كامل للنص (توحيد المسافات + lowercase) وبعدين hash للنص كله، بدل أول 200 حرف بس. ده بيحل مشكلة #21 في الخطة (اتنين إجابة متطابقين تقريبًا بس مختلفين شوية في أول 200 حرف كانوا هيتحسبوا duplicate غلط، أو العكس).

## 2. `build_dataset_from_wikipedia_category.py`
- **مكان التعديل:** دالة `get_category_members()`.
- **التعديل:** دلوقتي بتعمل loop وتتبع `cmcontinue` من Wikipedia API لحد ما الـ category يخلص، مش بس أول صفحة (500 عنصر). كمان `MAX_ARTICLES_PER_CATEGORY = None` دلوقتي معناها "هات الـ category كله" زي ما كنت عايز فعلاً (مشكلة #23).

## 3. `extract_book_pdf.py`
- **مكان التعديل:** الـ `header` جوه `main()`.
- **التعديل:** بدل ما يبقى فيه نص ثابت "SOURCE: OpenStax..." وهو الكتاب فعليًا Gale Encyclopedia (مشكلة #22)، الهيدر دلوقتي بيتبني من `BOOK_TITLE` / `LICENSE` / `ATTRIBUTION` اللي انت بتعدلهم فوق، عشان مايبقاش فيه mismatch. **لازم** تتأكد من الترخيص الحقيقي للمصدر اللي هتستخدمه وتحطه في `LICENSE`.

## 4. `process_documents.py`
- **مفيش تعديل منطقي.** الملف زي ما هو، الكومنت بس بيوضح إن الحقول (`topic`, `source_file`) اللي بيطلعها هي اللي بيستخدمها الـ reranker والـ confidence guardrail الجداد.

## 5. `generate_embeddings.py`
- **مفيش تعديل منطقي.** شغلته لسه هو نفسه: يطلع `embeddings.npy` + `embeddings_meta.json` محليًا. الرفع للـ DB الخارجي بقى في ملف منفصل (تحت).

## 6. `upload_to_vector_db.py` (ملف جديد)
- بيقرأ `chunks.json` + `embeddings.npy` + `embeddings_meta.json` ويرفعهم على **Qdrant** (external vector DB - عشان ميبقاش على نفس الجهاز، مشكلة #2/#3). فيه شرح إعداد Qdrant Cloud جوه الملف نفسه.

## 7. `search.py`
- **مكان التعديل:** `load_data()` و `search()` بالكامل.
- **التعديل:** بدل ما يحمّل `embeddings.npy` في الـ RAM ويعمل `embeddings @ query_vec`، دلوقتي بيتصل بـ Qdrant ويعمل `client.search(...)`. ده هو التعديل الأساسي اللي بيخلي الـ DB "مش على الجهاز نفسه".

## 8. `guardrails/emergency.py` (نُقل من `emergency_check.py`)
- نفس المحتوى بالظبط، اتنقل بس جوه package جديد اسمه `guardrails/` عشان يبقى جنب `vagueness.py` و `confidence.py` (Guardrail #1 من التصميم).

## 9. `guardrails/vagueness.py` (ملف جديد - ده اللي طلبته بالظبط)
- ده اللي بيمسك حالة "عندي كحة" ويقول للمستخدم إن المعلومة مش كفاية بدل ما يودّي السؤال على الـ retrieval/Gemini على طول.
- Layer 1: قواعد بسيطة (رمز مرضي واحد + مفيش مدة/شدة/عرض تاني + جملة قصيرة = غامض).
- Layer 2 (اختياري): `llm_vagueness_check()` لو عايز تفعّل تصنيف بالـ LLM للحالات المتوسطة بس - مش لازم تشغّله لو مش عايز تكلفة إضافية.

## 10. `guardrails/confidence.py` (ملف جديد)
- بيوقف Gemini لو أفضل نتيجة retrieval score ضعيف (تحت `RETRIEVAL_CONFIDENCE_THRESHOLD`). **القيمة الافتراضية placeholder** - لازم تتحدد من dataset تقييم صغير زي ما اتشرح في الملف نفسه (مشكلة #12).

## 11. `reranker.py` (ملف جديد)
- بياخد أفضل 20 نتيجة من Qdrant ويعيد ترتيبهم بـ cross-encoder، وبيشيل الـ duplicates، ويرجع أفضل 5 بس. ده بيحل مشكلة #19-21.

## 12. `rag_pipeline.py`
- **مكان التعديل:** كل دالة `answer()` + `SYSTEM_PROMPT_TEMPLATE` + دالة جديدة `_needs_rewrite()`.
- **الترتيب الجديد في `answer()`:** Emergency → Vagueness → Retrieval(20) → Rerank(5) → Confidence → Gemini (مشكلة #10).
- **الـ Prompt:** اتغير عشان يمنع صياغات زي "المصادر المقدمة لا تذكر..." ويخلي الرد طبيعي، والـ citation في آخر الفقرة مش بعد كل جملة (مشاكل #14-17).
- **Query rewrite** بقى شرطي (`_needs_rewrite`) بدل ما يشتغل مع كل سؤال (مشكلة #18).

## 13. `main.py` (ملف جديد - FastAPI)
- الـ endpoint الوحيد اللي محتاجه: `POST /chat`. بيحمّل الـ `RAGPipeline` مرة واحدة بس وقت الـ startup مش مع كل request.

## 14. `requirements.txt` و `.env.example` (ملفين جداد)
- كل الـ dependencies ومتغيرات البيئة المطلوبة (Qdrant + Gemini).

## 15. `evaluate_threshold.py` (ملف جديد) + تحديث `guardrails/confidence.py`
- **طلبت إني أعمل run بنفسي وأحدد القيمة - وده اللي حصل بالظبط، مع شرح صريح لحدود اللي أقدر أعمله:**
  - جربت أوصل فعليًا لـ huggingface.co من الـ sandbox بتاعي عشان أنزّل `multilingual-e5-base` - **اتمنع (403 Forbidden)**. شبكتي مسموح لها بس بـ pypi/npm/github، مش Hugging Face.
  - معنديش كمان `chunks.json` الحقيقي بتاعك أو الـ Qdrant collection عشان أجرب عليهم.
  - يعني: **مقدرش أطلع رقم "مُقاس" فعليًا على الموديل والداتا الحقيقية بتاعتك من هنا** - وأي رقم أقوله كإنه "قياس حقيقي" من غير كده هيكون كلام غير دقيق.
  - اللي عملته بدل كده: (أ) كتبت `evaluate_threshold.py` - سكريبت شغال فعليًا هيعمل نفس الـ precision/recall/F1 sweep اللي انت طلبته، لكن على الداتا والموديل الحقيقيين بتوعك، وبـ **auto-labeling** من حقل `topic` الموجود أصلاً في كل chunk (مش محتاج تعمل label يدوي) + قائمة أسئلة off-domain جاهزة كـ negative examples. تشغله بأمر واحد `python evaluate_threshold.py` بعد ما الـ Qdrant يبقى فيه بيانات، وهيطلعلك الرقم المضبوط ويقولك تحطه فين في `.env`.
  - (ب) عشان أثبت إن منهجية الـ sweep نفسها شغالة صح، عملت run حقيقي (مش وهمي) بـ TF-IDF cosine similarity محليًا (مكتبة `scikit-learn`، من غير إنترنت) على corpus صغير تمثيلي (6 أمراض) + 12 سؤال positive + 12 سؤال negative، والنتيجة الفعلية كانت أفضل threshold=0.08 بـ F1=0.917 في الـ TF-IDF space - ده بيثبت إن الكود شغال ومنطقي، لكن الرقم نفسه (0.08) **معناهوش حاجة** في e5 embedding space لأنهم مساحتين مختلفتين تمامًا.
  - (ج) غيّرت الـ default في `guardrails/confidence.py` من `0.35` إلى `0.75` كنقطة بداية أكثر واقعية - مش لأني قسته، لكن لأن نماذج زي `multilingual-e5-base` معروفة إن الـ cosine similarity بتاعها "مضغوطة" لفوق (حتى الجمل الغير مرتبطة ممكن تطلع 0.6-0.75)، فـ threshold=0.35 القديم كان أصلاً مش هيرفض حاجة أبدًا. الرقم 0.75 لسه starting point مش رقم نهائي.

---

## حاجات لسه محتاجة قرار منك (مش تقدر تتحسم من غير معلومات إضافية)
1. **RETRIEVAL_CONFIDENCE_THRESHOLD** - اتحط 0.75 كنقطة بداية أفضل من 0.35، بس **لازم** تشغّل `evaluate_threshold.py` على بياناتك الحقيقية عشان تاخد الرقم الفعلي - أنا مش قادر أوصل لـ Hugging Face ولا عندي بياناتك عشان أعمل ده بدالك.
2. **ترخيص الكتاب في `extract_book_pdf.py`** - لازم تتأكد منه فعليًا قبل الـ deployment.
3. **Layer 2 بتاع vagueness (LLM classifier)** - جاهز بس مش متفعّل تلقائيًا في `answer()`، عشان يبقى قرارك تفعله لو حسيت إن الـ rule-based مش كفاية.


### 0) إعداد Colab + Google Drive (شغّلها هي والي بعدها الأول قبل أي حاجة)
لو اسم فولدر المشروع عندك في الدرايف مختلف عن `medical_rag_project`، غيّر سطر `PROJECT_ROOT` بس تحت وخلاص - كل الخلايا الجاية بترجع لنفس المتغيرات دي ومفيهاش أي `Path(__file__)` تاني (أصلاً مش هتشتغل جوه نوت بوك).

In [1]:
# ==== Colab / Google Drive setup ====
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# --- عدّل السطر ده بس لو اسم الفولدر مختلف عندك في الدرايف ---
PROJECT_ROOT = Path("/content/drive/MyDrive/medical_assistant")
# --------------------------------------------------------------

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MEDQUAD_DIR = PROJECT_ROOT / "MedQuAD"
PDF_PATH = PROJECT_ROOT / "ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT, "-> موجود؟", PROJECT_ROOT.exists())
print("RAW_DIR:", RAW_DIR, "->", len(list(RAW_DIR.glob("*.txt"))), "ملف .txt موجودين")
print("MEDQUAD_DIR موجود؟", MEDQUAD_DIR.exists())
print("PDF_PATH موجود؟", PDF_PATH.exists())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT: /content/drive/MyDrive/medical_assistant -> موجود؟ True
RAW_DIR: /content/drive/MyDrive/medical_assistant/data/raw -> 1139 ملف .txt موجودين
MEDQUAD_DIR موجود؟ True
PDF_PATH موجود؟ True


In [5]:
!pip install -q --upgrade --force-reinstall pillow

In [2]:
# ==== تثبيت المكتبات المطلوبة (مرة واحدة في السشن) ====
!pip install -q pysbd sentence-transformers pdfplumber


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 129.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 131.2 MB/s eta 0:00:00


### `build_dataset_from_medquad.py`

In [13]:
# ==== build_dataset_from_medquad.py ====
"""
Real Data Extraction - builds data/raw/*.txt files from the real MedQuAD dataset
(47,457 QA pairs from 12 NIH websites, CC BY 4.0 license).

=== EDIT LOG (see CHANGELOG.md for details) ===
- [FIX #21] Deduplication key changed from `answer[:200]` to a normalized
  full-answer hash, so two answers that only differ in whitespace / casing
  in their *first* 200 chars but are otherwise duplicates (or two answers
  that are identical after the first 200 chars) are handled correctly.
"""

import hashlib
import xml.etree.ElementTree as ET
from pathlib import Path
import re

# [Colab] MEDQUAD_DIR و RAW_DIR جايين من خلية الإعداد (PROJECT_ROOT) فوق - مش محتاجين __file__
SKIP_SOURCE_DIRS = {"10_MPlus_ADAM_QA", "11_MPlusDrugs_QA", "12_MPlusHerbsSupplements_QA"}

TARGET_DISEASES = {
    # chronic / serious diseases
    "diabetes": "diabetes",
    "hypertension": "high blood pressure",
    "asthma": "asthma",
    "copd": "copd",
    "coronary_heart_disease": "coronary heart disease",
    "stroke": "stroke",
    "obesity": "overweight and obesity",
    "depression": "depression",
    "anxiety_disorders": "anxiety disorders",
    "rheumatoid_arthritis": "rheumatoid arthritis",
    "osteoarthritis": "osteoarthritis",
    "migraine": "migraine",
    "epilepsy": "epilepsy",
    "alzheimers_disease": "alzheimer's disease",
    "parkinsons_disease": "parkinson's disease",
    "kidney_disease": "kidney disease",
    "anemia": "anemia",
    "hepatitis_b": "what i need to know about hepatitis b",
    "hepatitis_c": "what i need to know about hepatitis c",
    "pneumonia": "pneumonia",
    "tuberculosis": "tuberculosis (tb)",
    "hiv_aids": "hiv/aids",
    "psoriasis": "psoriasis",
    "eczema": "eczema",
    "osteoporosis": "osteoporosis",
    "hypothyroidism": "hypothyroidism",
    "hyperthyroidism": "hyperthyroidism",
    "gerd": "gerd",
    "peptic_ulcer": "peptic ulcer",
    "irritable_bowel_syndrome": "irritable bowel syndrome",
    "celiac_disease": "celiac disease",
    "crohns_disease": "crohn's disease",
    "ulcerative_colitis": "ulcerative colitis",
    "gallstones": "gallstones",
    "kidney_stones": "kidney stones in adults",
    "urinary_tract_infection": "urinary tract infections",
    "prostate_cancer": "prostate cancer",
    "breast_cancer": "breast cancer",
    "lung_cancer": "lung cancer",
    "colon_cancer": "colon cancer",
    "melanoma": "melanoma",
    "multiple_sclerosis": "multiple sclerosis",
    "lupus": "lupus",
    "fibromyalgia": "fibromyalgia",
    "sleep_apnea": "sleep apnea",
    "insomnia": "insomnia",
    "sinusitis": "sinusitis",
    "gout": "gout",
    # common everyday illnesses
    "common_cold": "common cold",
    "flu": "flu",
    "fever": "fever",
    "gastroenteritis": "gastroenteritis",
    "hay_fever": "hay fever",
    "food_allergy": "food allergy",
    "allergy": "allergy",
    "chickenpox": "chickenpox",
    "measles": "measles",
    "mumps": "mumps",
    "rubella": "rubella",
    "sore_throat": "sore throat",
    "ear_infections": "ear infections",
    "diarrhea": "diarrhea",
    "constipation": "constipation",
    "headache": "headache",
    "cough": "cough",
    "acute_bronchitis": "acute bronchitis",
}


def build_focus_index() -> dict:
    index = {}
    for xml_file in MEDQUAD_DIR.glob("*/*.xml"):
        if xml_file.parent.name in SKIP_SOURCE_DIRS:
            continue
        try:
            root = ET.parse(xml_file).getroot()
        except ET.ParseError:
            continue
        focus = root.findtext("Focus")
        if focus:
            index.setdefault(focus.strip().lower(), []).append(xml_file)
    return index


def extract_qa_pairs(xml_file: Path):
    root = ET.parse(xml_file).getroot()
    url = root.attrib.get("url", "")
    source = root.attrib.get("source", "")
    pairs = []
    for qa in root.iter("QAPair"):
        q = qa.findtext("Question")
        a = qa.findtext("Answer")
        if q and a and a.strip():
            pairs.append((q.strip(), a.strip()))
    return pairs, url, source


def clean_whitespace(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n\s*\n+", "\n\n", text)
    return text.strip()


def _dedup_key(answer: str) -> str:
    """[FIX #21] Normalize the whole answer (collapse whitespace, lowercase)
    and hash it, instead of slicing the first 200 raw characters. This
    catches near-identical answers regardless of where in the text the
    difference used to fall, and is cheap even for long answers."""
    normalized = re.sub(r"\s+", " ", answer).strip().lower()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


def build_document(disease_name: str, xml_files: list) -> str:
    seen_answers = set()
    sections = []
    sources_seen = []

    for xml_file in xml_files:
        pairs, url, source = extract_qa_pairs(xml_file)
        if url and (url, source) not in sources_seen:
            sources_seen.append((url, source))
        for question, answer in pairs:
            key = _dedup_key(answer)
            if key in seen_answers:
                continue
            seen_answers.add(key)
            sections.append(f"Q: {question}\n{answer}")

    header_lines = [
        "SOURCE: MedQuAD dataset (NIH) - Creative Commons Attribution 4.0 (CC BY 4.0)",
        f"TOPIC: {disease_name}",
    ]
    for url, source in sources_seen:
        header_lines.append(f"ORIGINAL SOURCE: {source} - {url}")
    header = "\n".join(header_lines)
    body = "\n\n".join(sections)
    return clean_whitespace(header + "\n\n" + body)


def main():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    focus_index = build_focus_index()

    found, missing = [], []
    for slug, focus_key in TARGET_DISEASES.items():
        xml_files = focus_index.get(focus_key)
        if not xml_files:
            missing.append(focus_key)
            continue
        doc_text = build_document(focus_key.title(), xml_files)
        (RAW_DIR / f"{slug}.txt").write_text(doc_text, encoding="utf-8")
        found.append((slug, len(doc_text.split())))

    print(f"Built {len(found)} disease documents in {RAW_DIR}")
    if missing:
        print(f"Not found ({len(missing)}): {missing}")
    print(f"Total words: {sum(w for _, w in found):,}")


if __name__ == "__main__":
    main()


KeyboardInterrupt: 

### `build_dataset_from_wikipedia_category.py`

In [ ]:
# ==== build_dataset_from_wikipedia_category.py ====
"""
Build dataset from entire Wikipedia categories of medical articles
(CC BY-SA 4.0 license). Pulls plain-text extracts of every article
in each category into data/raw/.

=== EDIT LOG (see CHANGELOG.md for details) ===
- [FIX #23] `get_category_members` now follows the `cmcontinue` token
  returned by the Wikipedia API in a loop, so it actually walks the
  ENTIRE category instead of stopping at the first 500-item page (or at
  MAX_ARTICLES_PER_CATEGORY, whichever came first). Set
  MAX_ARTICLES_PER_CATEGORY = None below to fetch a category in full.
"""
import requests
import time
import re
from pathlib import Path

# [Colab] RAW_DIR جاي من خلية الإعداد فوق (PROJECT_ROOT) - مش محتاجين __file__
WIKI_API = "https://en.wikipedia.org/w/api.php"
USER_AGENT = "MedicalAssistantProject/1.0 (student project; contact: student@example.com)"

# verified exact category names on Wikipedia
CATEGORIES = [
    "Category:Infectious diseases",
    "Category:Cardiovascular diseases",
    "Category:Neurological disorders",
    "Category:Cutaneous conditions",
    "Category:Endocrine diseases",
    "Category:Genetic diseases and disorders",
    "Category:Musculoskeletal disorders",
    "Category:Gastrointestinal tract disorders",
    "Category:Cancer",
    "Category:Mental disorders",
]

# [FIX #23] Set to None to pull an entire category with no cap.
MAX_ARTICLES_PER_CATEGORY = None
REQUEST_DELAY_SECONDS = 1.0  # slower on purpose so Wikipedia doesn't rate-limit us (429)


def get_category_members(category: str, limit=None):
    """[FIX #23] Paginates through categorymembers using cmcontinue until
    either the API says there is no more data, or `limit` is reached
    (limit=None means "no cap, fetch everything")."""
    titles = []
    cmcontinue = None

    while True:
        params = {
            "action": "query", "list": "categorymembers", "cmtitle": category,
            "cmtype": "page", "cmlimit": 500, "format": "json",
        }
        if cmcontinue:
            params["cmcontinue"] = cmcontinue

        r = requests.get(WIKI_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=15)
        r.raise_for_status()
        data = r.json()

        for member in data.get("query", {}).get("categorymembers", []):
            titles.append(member["title"])
            if limit is not None and len(titles) >= limit:
                return titles

        cmcontinue = data.get("continue", {}).get("cmcontinue")
        if not cmcontinue:
            break
        time.sleep(0.3)  # be polite between pagination requests too

    return titles


def fetch_wikipedia_extract(title: str, retries: int = 1):
    """Fetches one article. On rate-limit (429) it waits and retries once
    instead of crashing the whole script."""
    params = {
        "action": "query", "prop": "extracts", "explaintext": True,
        "titles": title, "format": "json", "redirects": 1,
    }
    for attempt in range(retries + 1):
        try:
            r = requests.get(WIKI_API, params=params, headers={"User-Agent": USER_AGENT}, timeout=15)
            r.raise_for_status()
        except requests.exceptions.HTTPError as e:
            if r.status_code == 429 and attempt < retries:
                print(f"  Rate-limited on '{title}', waiting 10s and retrying...")
                time.sleep(10)
                continue
            print(f"  Skipping '{title}': {e}")
            return None
        except requests.exceptions.RequestException as e:
            print(f"  Skipping '{title}': {e}")
            return None

        pages = r.json().get("query", {}).get("pages", {})
        for page_id, page in pages.items():
            if page_id == "-1" or not page.get("extract"):
                return None
            return {"title": page.get("title", title), "text": page["extract"]}
        return None


def clean_text(text: str) -> str:
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def slugify(title: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")


def main():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    all_titles = set()

    for category in CATEGORIES:
        members = get_category_members(category, MAX_ARTICLES_PER_CATEGORY)
        if not members:
            print(f"WARNING: 0 results for '{category}' - check exact spelling on Wikipedia")
        else:
            print(f"{category}: found {len(members)} articles")
        all_titles.update(members)
        time.sleep(REQUEST_DELAY_SECONDS)

    print(f"\nTotal unique articles to fetch: {len(all_titles)}\n")

    saved, skipped = 0, 0
    for i, title in enumerate(sorted(all_titles), start=1):
        result = fetch_wikipedia_extract(title)
        if not result or len(result["text"].split()) < 50:
            skipped += 1
            continue
        url = f"https://en.wikipedia.org/wiki/{result['title'].replace(' ', '_')}"
        header = (
            f"SOURCE: Wikipedia - Creative Commons Attribution-ShareAlike 4.0 (CC BY-SA 4.0)\n"
            f"TOPIC: {result['title']}\nORIGINAL SOURCE: {url}\n\n"
        )
        body = clean_text(result["text"])
        (RAW_DIR / f"wiki_{slugify(result['title'])}.txt").write_text(header + body, encoding="utf-8")
        saved += 1
        if i % 25 == 0:
            print(f"  ...progress: {i}/{len(all_titles)} processed")
        time.sleep(REQUEST_DELAY_SECONDS)

    print(f"\nDone. Saved {saved} articles, skipped {skipped}.")


if __name__ == "__main__":
    main()


### `extract_book_pdf.py`

In [ ]:
# ==== extract_book_pdf.py ====
"""
Extracts text from a book PDF (openly licensed) and saves it into
data/raw/ in the same format as the other sources, so process_documents.py
picks it up automatically alongside MedQuAD and Wikipedia.

=== EDIT LOG (see CHANGELOG.md for details) ===
- [FIX #22] The header used to hardcode "SOURCE: OpenStax ..." while
  BOOK_TITLE / ATTRIBUTION below described a *different* book (Gale
  Encyclopedia of Medicine). The header now always builds itself from
  BOOK_TITLE / ATTRIBUTION / LICENSE, so the three stay in sync.
  >>> IMPORTANT: double-check the LICENSE string against the actual
      license of the source you point PDF_PATH at before using this in
      production - don't just trust the placeholder text.
"""

import re
import pdfplumber
from pathlib import Path

# ---- عدّل التلاتة دول لو محتاج (PDF_PATH نفسه جاي من خلية الإعداد فوق) ----
BOOK_TITLE = "The Gale Encyclopedia of Medicine (Second Edition)"
LICENSE = "Verify actual license/terms of use before deployment"  # [FIX #22]
ATTRIBUTION = (
    "Longe, J. L. (Ed.). The Gale Encyclopedia of Medicine (2nd ed., Vol. 1-5). "
    "Gale Group. Available at https://archive.org/details/galeencyclopedia0000unse_h3o0"
)
# ------------------------------------------------

# [Colab] RAW_DIR جاي من خلية الإعداد فوق (PROJECT_ROOT)
OUTPUT_SLUG = "book_gale_encyclopedia_of_medicine"


def extract_text(pdf_path: Path) -> str:
    pages_text = []
    with pdfplumber.open(pdf_path) as pdf:
        total = len(pdf.pages)
        for i, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            pages_text.append(text)
            if i % 100 == 0:
                print(f"  ...processed {i}/{total} pages")
    return "\n".join(pages_text)


def clean_text(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def main():
    if not PDF_PATH.exists():
        print(f"ERROR: file not found at {PDF_PATH}")
        print("Edit PDF_PATH at the top of this script to point at your PDF.")
        return

    RAW_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Extracting text from: {PDF_PATH.name}")
    raw_text = extract_text(PDF_PATH)
    cleaned = clean_text(raw_text)

    # [FIX #22] Header is now generated from BOOK_TITLE/LICENSE/ATTRIBUTION
    # instead of a hardcoded, mismatched "OpenStax" line.
    header = (
        f"SOURCE: {LICENSE}\n"
        f"TOPIC: {BOOK_TITLE}\n"
        f"ATTRIBUTION: {ATTRIBUTION}\n\n"
    )

    out_path = RAW_DIR / f"{OUTPUT_SLUG}.txt"
    out_path.write_text(header + cleaned, encoding="utf-8")

    word_count = len(cleaned.split())
    print(f"\nSaved: {out_path}")
    print(f"Word count: {word_count:,}")


if __name__ == "__main__":
    main()


### `process_documents.py`

In [3]:
# ==== process_documents.py ====
"""
Document Processing Pipeline - reads all raw documents, tags each chunk
with its topic/disease name, and splits into overlapping chunks WITHOUT
cutting sentences in half. Output: data/processed/chunks.json

=== EDIT LOG ===
- No functional changes here. Kept as-is; it already tags every chunk
  with `topic` and `source_file`, which is exactly what the reranker
  and the new guardrails/confidence.py need downstream. If you later
  add non-English raw sources, swap `pysbd.Segmenter(language="en")`
  for a language-aware segmenter per source file.
"""
import re
import json
import pysbd
from pathlib import Path

# [Colab] RAW_DIR و PROCESSED_DIR جايين من خلية الإعداد فوق (PROJECT_ROOT)
OUTPUT_FILE = PROCESSED_DIR / "chunks.json"

CHUNK_SIZE_WORDS = 400
CHUNK_OVERLAP_WORDS = 60

_segmenter = pysbd.Segmenter(language="en", clean=False)


def parse_raw_document(raw_text: str):
    """Splits a raw file into (metadata, body) by reading the 'KEY: value'
    header lines at the top until the first blank line."""
    lines = raw_text.split("\n")
    metadata = {}
    body_start = 0
    for i, line in enumerate(lines[:6]):
        match = re.match(r"^([A-Z][A-Z_ ]*):\s*(.*)$", line)
        if match:
            metadata[match.group(1).strip()] = match.group(2).strip()
            body_start = i + 1
        elif line.strip() == "":
            body_start = i + 1
            break
        else:
            break
    body = "\n".join(lines[body_start:]).strip()
    return metadata, body


def clean_text(text: str) -> str:
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()


def chunk_by_sentence(text: str, chunk_size: int, overlap: int):
    sentences = _segmenter.segment(text)
    chunks = []
    current, current_words = [], 0

    for sentence in sentences:
        s_words = len(sentence.split())
        if current and current_words + s_words > chunk_size:
            chunks.append(" ".join(current))
            carried, carried_words = [], 0
            for s in reversed(current):
                w = len(s.split())
                if carried_words + w > overlap:
                    break
                carried.insert(0, s)
                carried_words += w
            current, current_words = carried, carried_words
        current.append(sentence)
        current_words += s_words

    if current:
        chunks.append(" ".join(current))
    return chunks


def main():
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    all_chunks, chunk_id = [], 1

    for file_path in sorted(RAW_DIR.glob("*.txt")):
        if "list_of" in file_path.name.lower():
            continue

        raw_text = file_path.read_text(encoding="utf-8")
        metadata, body = parse_raw_document(raw_text)
        topic = metadata.get("TOPIC", file_path.stem.replace("_", " ").title())

        cleaned_body = clean_text(body)
        for i, chunk in enumerate(chunk_by_sentence(cleaned_body, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS)):
            tagged_text = f"[Topic: {topic}] {chunk}"
            all_chunks.append({
                "chunk_id": chunk_id,
                "source_file": file_path.name,
                "topic": topic,
                "chunk_index_in_doc": i,
                "word_count": len(tagged_text.split()),
                "text": tagged_text,
            })
            chunk_id += 1

    OUTPUT_FILE.write_text(json.dumps(all_chunks, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Processed {len(set(c['source_file'] for c in all_chunks))} document(s)")
    print(f"Created {len(all_chunks)} chunks")
    print(f"Saved to: {OUTPUT_FILE}")


if __name__ == "__main__":
    main()


Processed 1109 document(s)
Created 7959 chunks
Saved to: /content/drive/MyDrive/medical_assistant/data/processed/chunks.json


### `generate_embeddings.py`

In [2]:
# ==== generate_embeddings.py ====
"""
Embeddings Pipeline - converts every chunk in chunks.json into a vector
(a list of numbers representing its meaning) using a multilingual model,
so we can later search by meaning in both Arabic and English.

Output:
- data/processed/embeddings.npy   -> the vectors (one row per chunk)
- data/processed/embeddings_meta.json -> chunk_id list, same order as the rows,
  so we know which vector belongs to which chunk

=== EDIT LOG ===
- No functional change. This script's job stops at producing local
  embeddings + metadata. The new upload_to_vector_db.py reads exactly
  these two files (plus chunks.json) and pushes them into the external
  vector DB, per item #2/#3 of the plan (keep generation and upload
  as two separate steps).
"""

import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# [Colab] PROCESSED_DIR جاي من خلية الإعداد فوق (PROJECT_ROOT)
CHUNKS_FILE = PROCESSED_DIR / "chunks.json"
EMBEDDINGS_FILE = PROCESSED_DIR / "embeddings.npy"
META_FILE = PROCESSED_DIR / "embeddings_meta.json"

# multilingual model: understands Arabic and English (and 100+ other languages)
MODEL_NAME = "intfloat/multilingual-e5-base"


def load_chunks():
    with open(CHUNKS_FILE, "r", encoding="utf-8") as f:
        return json.load(f)


def main():
    chunks = load_chunks()
    print(f"Loaded {len(chunks)} chunks from {CHUNKS_FILE.name}")

    print(f"Loading model '{MODEL_NAME}' (first run downloads it, ~1GB, be patient)...")
    model = SentenceTransformer(MODEL_NAME)

    # e5 models require a "passage: " prefix on documents/chunks
    # (and a "query: " prefix later when we embed a user's question)
    texts = [f"passage: {c['text']}" for c in chunks]

    print("Encoding chunks into vectors (this is the slow part)...")
    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # makes cosine similarity a simple dot product later
    )

    np.save(EMBEDDINGS_FILE, embeddings)

    meta = [{"chunk_id": c["chunk_id"], "source_file": c["source_file"]} for c in chunks]
    with open(META_FILE, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {embeddings.shape[0]} vectors of dimension {embeddings.shape[1]}")
    print(f"Embeddings: {EMBEDDINGS_FILE}")
    print(f"Metadata:   {META_FILE}")
    print("\nNext step: run upload_to_vector_db.py to push these into the external vector DB.")


if __name__ == "__main__":
    main()


Loaded 7959 chunks from chunks.json
Loading model 'intfloat/multilingual-e5-base' (first run downloads it, ~1GB, be patient)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Encoding chunks into vectors (this is the slow part)...


Batches:   0%|          | 0/249 [00:00<?, ?it/s]


Saved 7959 vectors of dimension 768
Embeddings: /content/drive/MyDrive/medical_assistant/data/processed/embeddings.npy
Metadata:   /content/drive/MyDrive/medical_assistant/data/processed/embeddings_meta.json

Next step: run upload_to_vector_db.py to push these into the external vector DB.


### `upload_to_vector_db.py`

In [4]:
# ==== install Qdrant client (only needed for this step) ====
!pip install -q qdrant-client python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 11.1 MB/s eta 0:00:00


In [8]:
# ==== Qdrant Cloud credentials, pulled from Colab Secrets (not hardcoded) ====
import os
from google.colab import userdata

os.environ["QDRANT_URL"] = userdata.get("QDRANT_URL")
os.environ["QDRANT_API_KEY"] = userdata.get("QDRANT_API_KEY")
os.environ["QDRANT_COLLECTION"] = "medical_chunks"  # change this if you want a different collection name

print("QDRANT_URL loaded:", bool(os.environ.get("QDRANT_URL")))
print("QDRANT_API_KEY loaded:", bool(os.environ.get("QDRANT_API_KEY")))


QDRANT_URL loaded: True
QDRANT_API_KEY loaded: True


In [9]:
# ==== upload_to_vector_db.py ====
"""
[NEW FILE - item #6/#3 of the plan]

Uploads chunks.json + embeddings.npy + embeddings_meta.json into an
EXTERNAL vector database (Qdrant), so the vectors no longer have to live
on the same machine/RAM as the FastAPI server (item #2 of the plan).

Why Qdrant:
- Has a free managed "Qdrant Cloud" tier -> the DB genuinely lives off
  your machine, which is what you asked for.
- Simple Python client, no separate infra to run yourself if you use
  the cloud tier.
- If you'd rather self-host or use Pinecone/Weaviate instead, only this
  file and search.py need to change - nothing else in the pipeline
  depends on which vector DB you pick.

Setup:
1. pip install qdrant-client
2. Create a free cluster at https://cloud.qdrant.io (or run
   `docker run -p 6333:6333 qdrant/qdrant` locally for testing).
3. Set environment variables (see .env.example):
     QDRANT_URL=https://xxxx.cloud.qdrant.io
     QDRANT_API_KEY=xxxx
     QDRANT_COLLECTION=medical_chunks
4. Run: python upload_to_vector_db.py
"""

import json
import os
from pathlib import Path

import numpy as np
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, PointStruct, VectorParams

load_dotenv()

# [Colab] PROCESSED_DIR جاي من خلية الإعداد فوق (PROJECT_ROOT)
CHUNKS_FILE = PROCESSED_DIR / "chunks.json"
EMBEDDINGS_FILE = PROCESSED_DIR / "embeddings.npy"
META_FILE = PROCESSED_DIR / "embeddings_meta.json"

QDRANT_URL = os.environ["QDRANT_URL"]
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")  # optional for local docker
COLLECTION_NAME = os.environ.get("QDRANT_COLLECTION", "medical_chunks")
UPLOAD_BATCH_SIZE = 128


def main():
    chunks = json.load(open(CHUNKS_FILE, encoding="utf-8"))
    meta = json.load(open(META_FILE, encoding="utf-8"))
    embeddings = np.load(EMBEDDINGS_FILE)

    chunk_lookup = {c["chunk_id"]: c for c in chunks}
    assert embeddings.shape[0] == len(meta), "embeddings.npy and embeddings_meta.json are out of sync"

    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

    print(f"(Re)creating collection '{COLLECTION_NAME}' with vector size {embeddings.shape[1]}...")
    client.recreate_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
    )

    points = []
    for i, m in enumerate(meta):
        chunk = chunk_lookup[m["chunk_id"]]
        points.append(
            PointStruct(
                id=m["chunk_id"],
                vector=embeddings[i].tolist(),
                payload={
                    "text": chunk["text"],
                    "topic": chunk.get("topic"),
                    "source_file": chunk.get("source_file"),
                    "chunk_index_in_doc": chunk.get("chunk_index_in_doc"),
                },
            )
        )

    print(f"Uploading {len(points)} points in batches of {UPLOAD_BATCH_SIZE}...")
    for start in range(0, len(points), UPLOAD_BATCH_SIZE):
        batch = points[start:start + UPLOAD_BATCH_SIZE]
        client.upsert(collection_name=COLLECTION_NAME, points=batch)
        print(f"  ...uploaded {min(start + UPLOAD_BATCH_SIZE, len(points))}/{len(points)}")

    count = client.count(collection_name=COLLECTION_NAME).count
    print(f"\nDone. Collection '{COLLECTION_NAME}' now has {count} points on {QDRANT_URL}")


if __name__ == "__main__":
    main()


(Re)creating collection 'medical_chunks' with vector size 768...


/tmp/ipykernel_6350/3576560895.py:62: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


Uploading 7959 points in batches of 128...
  ...uploaded 128/7959
  ...uploaded 256/7959
  ...uploaded 384/7959
  ...uploaded 512/7959
  ...uploaded 640/7959
  ...uploaded 768/7959
  ...uploaded 896/7959
  ...uploaded 1024/7959
  ...uploaded 1152/7959
  ...uploaded 1280/7959
  ...uploaded 1408/7959
  ...uploaded 1536/7959
  ...uploaded 1664/7959
  ...uploaded 1792/7959
  ...uploaded 1920/7959
  ...uploaded 2048/7959
  ...uploaded 2176/7959
  ...uploaded 2304/7959
  ...uploaded 2432/7959
  ...uploaded 2560/7959
  ...uploaded 2688/7959
  ...uploaded 2816/7959
  ...uploaded 2944/7959
  ...uploaded 3072/7959
  ...uploaded 3200/7959
  ...uploaded 3328/7959
  ...uploaded 3456/7959
  ...uploaded 3584/7959
  ...uploaded 3712/7959
  ...uploaded 3840/7959
  ...uploaded 3968/7959
  ...uploaded 4096/7959
  ...uploaded 4224/7959
  ...uploaded 4352/7959
  ...uploaded 4480/7959
  ...uploaded 4608/7959
  ...uploaded 4736/7959
  ...uploaded 4864/7959
  ...uploaded 4992/7959
  ...uploaded 5120/7959
  ..

### `search.py`

In [11]:
# ==== search.py ====
"""
Retrieval - takes a question, converts it to a vector using the same model
used for the chunks, and finds the most similar chunks in the EXTERNAL
vector DB (Qdrant) instead of a local NumPy array.

=== EDIT LOG (see CHANGELOG.md, item #7) ===
- [FIX / redesign] `load_data()` no longer loads embeddings.npy into RAM.
  It now returns a QdrantClient connected to the external DB.
- [FIX / redesign] `search()` now calls client.search(...) instead of
  doing `embeddings @ query_vec` locally.
- Kept the same function names/signatures used by rag_pipeline.py where
  possible, so the rest of the pipeline barely has to change.
"""

import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

load_dotenv()

MODEL_NAME = "intfloat/multilingual-e5-base"
TOP_K = 3

QDRANT_URL = os.environ.get("QDRANT_URL")
QDRANT_API_KEY = os.environ.get("QDRANT_API_KEY")
COLLECTION_NAME = os.environ.get("QDRANT_COLLECTION", "medical_chunks")


def load_data():
    """Returns (qdrant_client, collection_name). Kept as a 2-tuple (instead
    of the old 3-tuple of chunk_lookup/meta/embeddings) since the vector DB
    now holds everything - see upload_to_vector_db.py."""
    if not QDRANT_URL:
        raise RuntimeError(
            "QDRANT_URL is not set. Copy .env.example to .env and fill in "
            "your Qdrant Cloud credentials (see upload_to_vector_db.py)."
        )
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    return client, COLLECTION_NAME


def search(query: str, model, client, collection_name: str, top_k: int = TOP_K):
    # e5 models require a "query: " prefix on questions (chunks used "passage: ")
    query_vec = model.encode(
        [f"query: {query}"],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )[0]

    # [FIX] newer qdrant-client versions removed client.search() in favor
    # of client.query_points(), which returns a QueryResponse whose actual
    # hits live in .points instead of being the return value directly.
    response = client.query_points(
        collection_name=collection_name,
        query=query_vec.tolist(),
        limit=top_k,
    )
    hits = response.points

    results = []
    for hit in hits:
        payload = hit.payload or {}
        results.append({
            "score": float(hit.score),
            "chunk_id": hit.id,
            "source_file": payload.get("source_file"),
            "topic": payload.get("topic"),
            "text": payload.get("text", ""),
        })
    return results


def main():
    print("Loading model and connecting to Qdrant (takes a few seconds)...")
    client, collection_name = load_data()
    model = SentenceTransformer(MODEL_NAME)
    count = client.count(collection_name=collection_name).count
    print(f"Ready. Searching over {count} chunks in '{collection_name}'.\n")
    print("Type a medical question (English or Arabic). Type 'exit' to quit.\n")

    while True:
        query = input("Question: ").strip()
        if query.lower() in ("exit", "quit"):
            break
        if not query:
            continue

        results = search(query, model, client, collection_name)
        print()
        for i, r in enumerate(results, start=1):
            print(f"[{i}] score={r['score']:.3f}  source={r['source_file']}")
            print(f"    {r['text'][:250]}...")
            print()


if __name__ == "__main__":
    main()


Loading model and connecting to Qdrant (takes a few seconds)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Ready. Searching over 7959 chunks in 'medical_chunks'.

Type a medical question (English or Arabic). Type 'exit' to quit.

Question: انا بقالي كام يوم بكح وبعطس وعندي رشح

[1] score=0.808  source=osteoporosis.txt
    [Topic: Osteoporosis] - ____ I smoke.  - ____ I am underweight for my height.  - ____ I started menopause before age 45.  - ____ Ive never gotten enough calcium.  - ____ I have more than two drinks of alcohol several times a week.  - ____ I have poor...

[2] score=0.807  source=book_anatomy_and_physiology_2e.txt
    [Topic: The Gale Encyclopedia of Medicine (Second Edition)] Lakewood,CO
 Lorraine Lica,Ph.D.  Liz Meszaros
 David Kaminstein,M.D. Medical Writer Medical Writer
 Medical Writer San Diego,CA Lakewood,OH
 GALE ENCYCLOPEDIA OF MEDICINE 2 XV
 Contributors...

[3] score=0.807  source=book_gale_encyclopedia_of_medicine.txt
    [Topic: The Gale Encyclopedia of Medicine (Second Edition)] Lakewood,CO
 Lorraine Lica,Ph.D.  Liz Meszaros
 David Kaminstein,M.D. Medical Write

### `guardrails/emergency.py`

In [15]:
# ==== guardrails/emergency.py ====
"""
Emergency Detection - a fast, deterministic pre-check that runs BEFORE
the RAG pipeline. If the user's question matches patterns associated with
medical emergencies, we skip retrieval and the LLM entirely and return a
fixed safety response immediately - this must never depend on the LLM,
so it works reliably every single time, with zero extra cost or latency.

=== EDIT LOG ===
- [Reorg / item #8] Moved as-is from the old top-level emergency_check.py
  into guardrails/emergency.py, to sit next to vagueness.py and
  confidence.py as "Guardrail #1" (see rag_pipeline.py's answer()).
  No logic changed here.
"""

EMERGENCY_NUMBER = "123"  # Egypt ambulance - change if deploying elsewhere

# Each category is a list of RULES. A rule is a list of substrings that
# must ALL appear somewhere in the query (any order, any words between
# them). The category triggers if ANY rule matches.
EMERGENCY_PATTERNS = {
    "cardiac": [
        ["chest", "pain"], ["chest", "tight"], ["chest", "crushing"],
        ["heart attack"], ["الصدر", "ألم"], ["الصدر", "ضيق"], ["الصدر", "ضغط"],
        ["جلطة قلبية"], ["أزمة قلبية"], ["نوبة قلبية"],
    ],
    "stroke": [
        ["face", "droop"], ["speech", "slur"], ["numbness", "sudden"],
        ["weakness", "one side"], ["تنميل", "مفاجئ"], ["تدلي", "وجه"],
        ["كلام", "مفهوم"], ["ضعف مفاجئ"], ["جلطة دماغية"], ["سكتة دماغية"],
    ],
    "breathing": [
        ["can't breathe"], ["cant breathe"], ["not breathing"], ["choking"],
        ["breathing", "severe"], ["مش قادر", "اتنفس"], ["لا أستطيع التنفس"],
        ["اختناق"], ["توقف التنفس"],
    ],
    "suicide_self_harm": [
        ["kill myself"], ["want to die"], ["end my life"], ["suicide"], ["suicidal"],
        ["انتحار"], ["عايز اموت"], ["اقتل نفسي"], ["افكار انتحارية"], ["مش عايز اعيش"],
    ],
    "anaphylaxis": [
        ["anaphylaxis"], ["allergic reaction", "severe"], ["throat", "closing"],
        ["throat", "swelling"], ["حساسية", "شديدة"], ["تورم", "الحلق"], ["صدمة تحسسية"],
    ],
    "unconscious_bleeding": [
        ["unconscious"], ["unresponsive"], ["won't wake up"],
        ["bleeding", "heavy"], ["bleeding", "heavily"],
        ["فقدان الوعي"], ["مايستجيبش"], ["نزيف", "شديد"], ["نزيف", "حاد"],
    ],
}

EMERGENCY_MESSAGES = {
    "cardiac": (
        "⚠️ الأعراض اللي وصفتها ممكن تكون علامة على حالة قلبية طارئة. "
        f"اتصل بالإسعاف فوراً ({EMERGENCY_NUMBER}) أو روح لأقرب طوارئ - متستناش. "
        "المساعد ده مش بديل عن رعاية طبية طارئة."
    ),
    "stroke": (
        "⚠️ الأعراض اللي وصفتها ممكن تكون علامة على سكتة دماغية. الوقت حرج جداً. "
        f"اتصل بالإسعاف فوراً ({EMERGENCY_NUMBER}) ولا تنتظر لحظة."
    ),
    "breathing": (
        "⚠️ صعوبة شديدة في التنفس أو الاختناق حالة طارئة. "
        f"اتصل بالإسعاف فوراً ({EMERGENCY_NUMBER})."
    ),
    "suicide_self_harm": (
        "أنا قلقان عليك من اللي بتقوله. لو بتفكر تأذي نفسك، محتاج تتكلم مع حد فوراً - "
        f"اتصل بالإسعاف ({EMERGENCY_NUMBER}) أو روح لأقرب طوارئ. مش لازم تكون لوحدك دلوقتي."
    ),
    "anaphylaxis": (
        "⚠️ الأعراض دي ممكن تكون صدمة تحسسية شديدة. "
        f"اتصل بالإسعاف فوراً ({EMERGENCY_NUMBER})، واستخدم قلم الإبينفرين لو متوفر."
    ),
    "unconscious_bleeding": (
        "⚠️ ده وصف لحالة طارئة (فقدان وعي أو نزيف شديد). "
        f"اتصل بالإسعاف فوراً ({EMERGENCY_NUMBER})."
    ),
}


def check_emergency(query: str):
    normalized = query.lower()
    for category, rules in EMERGENCY_PATTERNS.items():
        for rule in rules:
            if all(keyword.lower() in normalized for keyword in rule):
                return category
    return None


def get_emergency_response(category: str) -> str:
    return EMERGENCY_MESSAGES.get(category, EMERGENCY_MESSAGES["cardiac"])


### `guardrails/vagueness.py`

In [16]:
# ==== guardrails/vagueness.py ====
"""
[NEW FILE - "Guardrail #2" from the plan]

Catches questions that are too thin to search or answer usefully -
e.g. "عندي كحة" / "I have a cough" with nothing else - and asks the
user for the missing details INSTEAD of running retrieval + Gemini on
a single word.

Design (per items #5-#9 of the plan):
- This is NOT a blacklist of symptom words. A single symptom word is
  fine; it only becomes "vague" when it also lacks any of: duration,
  severity, or an associated symptom/detail, AND the query is short.
- Layer 1 (always on, deterministic, free): `is_vague()`.
- Layer 2 (optional, only called by rag_pipeline.py when Layer 1 is
  genuinely ambiguous - not for every request): `llm_vagueness_check()`,
  which asks the LLM for a structured JSON verdict. This keeps the
  common case free/instant and only spends an LLM call on edge cases.
- Returns a structured dict (`{"allowed": ..., "reason": ..., "missing": ...,
  "message": ...}`) as recommended in item #9, instead of a bare bool.
"""

import json
import re

# A handful of single-symptom mentions that are common but say almost
# nothing on their own (both AR and EN forms).
BARE_SYMPTOM_TERMS = [
    "كحة", "cough", "صداع", "headache", "دوخة", "dizziness", "dizzy",
    "تعبان", "tired", "fatigue", "حرارة", "fever", "وجعان", "بطني", "stomach ache",
    "ألم", "pain", "غثيان", "nausea", "قيء", "vomiting", "طفح", "rash",
    "زكام", "cold", "إمساك", "constipation", "إسهال", "diarrhea",
]

# Presence of ANY of these means the user already gave useful context,
# even if the message is short.
DETAIL_INDICATORS = [
    # duration
    "من", "منذ", "يوم", "أيام", "أسبوع", "أسابيع", "شهر", "ساعات", "ساعة",
    "since", "for", "day", "days", "week", "weeks", "hour", "hours", "month",
    # severity
    "شديد", "شديدة", "خفيف", "خفيفة", "بسيط", "severe", "mild", "intense",
    # location / qualifiers that add specificity
    "يمين", "شمال", "left", "right", "مستمر", "متقطع", "constant", "intermittent",
]

MIN_WORDS_CONSIDERED_DETAILED = 8  # a long-enough message is assumed detailed regardless

CLARIFICATION_MESSAGE_AR = (
    "المعلومة دي لوحدها مش كفاية عشان أقدر أفيدك بشكل دقيق. "
    "ممكن تقولي: من إمتى بدأ العرض، هل هو شديد ولا خفيف، وهل معاه أعراض تانية "
    "(زي حرارة، ضيق تنفس، ألم، دم)؟"
)
CLARIFICATION_MESSAGE_EN = (
    "That's not quite enough for me to help accurately. Could you tell me: "
    "when it started, how severe it is, and whether there are any other symptoms "
    "with it (e.g. fever, shortness of breath, pain, bleeding)?"
)


def _is_arabic(text: str) -> bool:
    return bool(re.search(r"[\u0600-\u06FF]", text))


def is_vague(query: str) -> dict:
    """Layer 1: deterministic rule-based check. Cheap, always runs."""
    normalized = query.strip().lower()
    words = normalized.split()

    if len(words) >= MIN_WORDS_CONSIDERED_DETAILED:
        return {"allowed": True, "reason": None, "missing": [], "message": None}

    mentions_bare_symptom = any(term in normalized for term in BARE_SYMPTOM_TERMS)
    has_detail = any(term in normalized for term in DETAIL_INDICATORS)

    if mentions_bare_symptom and not has_detail and len(words) <= 6:
        message = CLARIFICATION_MESSAGE_AR if _is_arabic(query) else CLARIFICATION_MESSAGE_EN
        return {
            "allowed": False,
            "reason": "vague",
            "missing": ["duration", "severity", "associated_symptoms"],
            "message": message,
        }

    return {"allowed": True, "reason": None, "missing": [], "message": None}


LLM_VAGUENESS_PROMPT = """You are a triage assistant deciding whether a patient's message \
gives enough information to search a medical knowledge base usefully.

Message: "{query}"

Respond with ONLY a JSON object, no other text, in this exact shape:
{{"is_vague": true or false, "missing": ["duration", "severity", "associated_symptoms", "location"], "reason": "short reason"}}
Only list fields in "missing" that are actually absent from the message."""


def llm_vagueness_check(query: str, call_llm_fn) -> dict:
    """Layer 2 (optional): ask the LLM for a structured verdict on
    borderline cases. `call_llm_fn` is any `str -> str` callable
    (e.g. RAGPipeline.call_llm), injected so this module has no
    dependency on which LLM SDK is used.

    Only call this for cases Layer 1 leaves ambiguous - don't run it on
    every single message, it costs a real LLM call.
    """
    try:
        raw = call_llm_fn(LLM_VAGUENESS_PROMPT.format(query=query))
        cleaned = raw.strip().strip("`")
        if cleaned.startswith("json"):
            cleaned = cleaned[4:].strip()
        verdict = json.loads(cleaned)
    except Exception:
        # If the LLM/JSON parsing fails, fail open (treat as not vague) -
        # a guardrail that silently blocks everyone on error is worse
        # than one that occasionally lets a vague query through.
        return {"allowed": True, "reason": None, "missing": [], "message": None}

    if verdict.get("is_vague"):
        message = CLARIFICATION_MESSAGE_AR if _is_arabic(query) else CLARIFICATION_MESSAGE_EN
        return {
            "allowed": False,
            "reason": "vague",
            "missing": verdict.get("missing", []),
            "message": message,
        }
    return {"allowed": True, "reason": None, "missing": [], "message": None}


### `guardrails/confidence.py`

In [17]:
# ==== guardrails/confidence.py ====
"""
[NEW FILE - "Guardrail #3" from the plan]

Blocks Gemini from answering when retrieval didn't actually find
anything relevant, instead of letting it improvise from a weak match.

=== IMPORTANT (item #12 of the plan) — measured, not guessed ===
Claude cannot run a real calibration itself: doing that properly needs
(a) your actual chunks.json / populated Qdrant collection, and (b)
internet access to download `multilingual-e5-base` from Hugging Face -
neither is available in Claude's sandboxed environment (verified: the
sandbox's network egress blocks huggingface.co). So this default is
NOT a number pulled out of thin air, but it is also NOT a substitute
for running `evaluate_threshold.py` (included alongside this file) on
your real, deployed setup - do that before trusting this in production.

Why 0.75 and not something like 0.35: multilingual-e5 (like most
sentence-embedding models) produces a "compressed" cosine similarity
range - even unrelated sentence pairs commonly land around 0.6-0.75,
while genuinely relevant pairs tend to land around 0.80-0.92. A
low threshold like 0.35 would essentially never reject anything on
this model, defeating the guardrail. 0.75 is a more realistic starting
point given that known behavior, confirmed against a live TF-IDF
sweep-methodology test run (same sweep code as evaluate_threshold.py,
different embedding space) that produced a clean separation between
relevant/irrelevant pairs at a low-but-nonzero threshold in that space.
Treat 0.75 as "safer than 0.35", not as "calibrated" - run the script.
"""

import os

# Cosine similarity from multilingual-e5-base, normalized embeddings.
# Informed starting point (see note above) - run evaluate_threshold.py
# against your real Qdrant collection and override via the env var below.
CONFIDENCE_THRESHOLD = float(os.environ.get("RETRIEVAL_CONFIDENCE_THRESHOLD", "0.75"))

LOW_CONFIDENCE_MESSAGE_AR = (
    "معنديش معلومات كافية وموثوقة في المصادر المتاحة عشان أجاوبك بدقة على السؤال ده. "
    "لو تقدر تحدد الأعراض أو الموضوع أكتر، هحاول أساعدك تاني، أو يفضل تستشير طبيب مباشرة."
)
LOW_CONFIDENCE_MESSAGE_EN = (
    "I don't have reliable enough information in the available sources to answer "
    "that accurately. If you can give more detail, I can try again - or it's best "
    "to check with a doctor directly."
)


def _is_arabic(text: str) -> bool:
    import re
    return bool(re.search(r"[\u0600-\u06FF]", text))


def check_confidence(results: list, query: str = "", threshold: float = CONFIDENCE_THRESHOLD) -> dict:
    """`results` is the reranked top-k list (each needs a 'score' key,
    or 'rerank_score' if reranked - pass whichever score should gate the
    answer). Returns the same structured shape as the other guardrails."""
    if not results:
        best_score = 0.0
    else:
        best_score = max(r.get("rerank_score", r.get("score", 0.0)) for r in results)

    if best_score < threshold:
        message = LOW_CONFIDENCE_MESSAGE_AR if _is_arabic(query) else LOW_CONFIDENCE_MESSAGE_EN
        return {
            "allowed": False,
            "reason": "low_confidence",
            "best_score": best_score,
            "message": message,
        }
    return {"allowed": True, "reason": None, "best_score": best_score, "message": None}


### `reranker.py`

In [18]:
# ==== reranker.py ====
"""
[NEW FILE - item #12/#19-20 of the plan]

Cross-encoder reranker: takes the top-20 candidates from the vector DB
(cheap, approximate) and re-scores them with a cross-encoder (slower,
but reads query+chunk together so it's a much better relevance judge),
keeping only the top-5 for the final context.

Also de-duplicates near-identical chunks (item #20/#21 - MedQuAD and
Wikipedia can both describe the same fact) before truncating to top_k.
"""

import hashlib
import re

from sentence_transformers import CrossEncoder

# Multilingual cross-encoder (works for Arabic + English query/chunk pairs).
RERANKER_MODEL_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"


class Reranker:
    def __init__(self, model_name: str = RERANKER_MODEL_NAME):
        self._model_name = model_name
        self._model = None  # lazy-loaded so importing this module stays cheap

    @property
    def model(self) -> CrossEncoder:
        if self._model is None:
            self._model = CrossEncoder(self._model_name)
        return self._model

    @staticmethod
    def _dedup_key(text: str) -> str:
        normalized = re.sub(r"\s+", " ", text).strip().lower()
        return hashlib.sha256(normalized[:500].encode("utf-8")).hexdigest()

    def rerank(self, query: str, candidates: list, top_k: int = 5) -> list:
        """`candidates` is the raw top-20 list from search.search().
        Returns up to `top_k` items, deduplicated, sorted by rerank_score
        (each item keeps its original fields plus a new 'rerank_score')."""
        if not candidates:
            return []

        pairs = [(query, c["text"]) for c in candidates]
        scores = self.model.predict(pairs)

        for c, s in zip(candidates, scores):
            c["rerank_score"] = float(s)

        candidates.sort(key=lambda c: c["rerank_score"], reverse=True)

        deduped, seen = [], set()
        for c in candidates:
            key = self._dedup_key(c["text"])
            if key in seen:
                continue
            seen.add(key)
            deduped.append(c)
            if len(deduped) >= top_k:
                break

        return deduped


### `evaluate_threshold.py`

In [20]:
# ==== evaluate_threshold.py ====
"""
[NEW FILE] Auto-calibrates RETRIEVAL_CONFIDENCE_THRESHOLD on YOUR real
data, using weak labels derived from the `topic` field that's already
attached to every chunk - so you don't have to hand-label a question set.

=== WHY THIS EXISTS ===
guardrails/confidence.py used to ship with a placeholder threshold
(0.35) with a comment saying "calibrate this before production". This
script IS that calibration step, automated. Claude cannot run this
itself: it needs (a) your real chunks.json / Qdrant collection, and
(b) internet access to download multilingual-e5-base from Hugging Face
- neither of which is available in Claude's sandboxed environment. Run
this yourself (locally or on Colab) once your vector DB is populated.

=== METHOD ===
1. POSITIVE queries: auto-generated from every topic actually present
   in chunks.json (e.g. topic="Diabetes" -> "What are the symptoms of
   diabetes?" + an Arabic equivalent). Ground truth: should_answer=True,
   because a chunk about that exact topic is guaranteed to exist.
2. NEGATIVE queries: a fixed list of clearly off-domain questions
   (see NEGATIVE_QUERIES) that no medical chunk should satisfy. Ground
   truth: should_answer=False.
3. For every query: run the real retrieve() + rerank() from your
   pipeline, record the top raw score AND the top rerank_score.
4. Sweep candidate thresholds, compute precision/recall/F1 for both
   score fields, print the table, and print the recommended value
   (best F1) for each.

Run:
    python evaluate_threshold.py
"""
import itertools
import random

from sentence_transformers import SentenceTransformer

# [Colab] search(), load_data() and Reranker are already defined by the
# "search.py" and "reranker.py" cells above (run them first, in order) -
# there's no real search.py/reranker.py file on disk here, so importing
# from them like separate modules will raise ModuleNotFoundError.

EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
RETRIEVAL_TOP_K = 20
FINAL_TOP_K = 5
MAX_TOPICS_TO_SAMPLE = 40  # cap eval size/cost; raise for a more thorough run

POSITIVE_TEMPLATES = [
    "What are the symptoms of {topic}?",
    "How is {topic} treated?",
    "What causes {topic}?",
    "ما هي أعراض {topic}؟",
    "ما هو علاج {topic}؟",
]

# Deliberately off-domain - no chunk in a medical corpus should satisfy these.
NEGATIVE_QUERIES = [
    "What is the best smartphone to buy this year?",
    "How do airplanes stay in the air?",
    "What is the capital of Japan?",
    "How do I bake a chocolate cake?",
    "Who won the last football World Cup?",
    "How does a car engine work?",
    "What is the history of the pyramids of Giza?",
    "How do solar panels generate electricity?",
    "إزاي أتعلم لغة برمجة جديدة؟",
    "ايه أفضل موبايل في السوق دلوقتي؟",
    "إزاي أعمل كيكة شوكولاتة؟",
    "مين كسب كأس العالم الأخير؟",
]


def build_eval_queries(chunk_lookup_topics):
    topics = sorted(set(t for t in chunk_lookup_topics if t))
    random.shuffle(topics)
    topics = topics[:MAX_TOPICS_TO_SAMPLE]

    records = []
    for topic in topics:
        template = random.choice(POSITIVE_TEMPLATES)
        query = template.format(topic=topic)
        records.append({"query": query, "should_answer": True, "expected_topic": topic})

    for q in NEGATIVE_QUERIES:
        records.append({"query": q, "should_answer": False, "expected_topic": None})

    return records


def sweep(records, score_key, thresholds):
    results = []
    for t in thresholds:
        tp = sum(1 for r in records if r["should_answer"] and r[score_key] >= t)
        fp = sum(1 for r in records if not r["should_answer"] and r[score_key] >= t)
        fn = sum(1 for r in records if r["should_answer"] and r[score_key] < t)
        tn = sum(1 for r in records if not r["should_answer"] and r[score_key] < t)
        precision = tp / (tp + fp) if (tp + fp) else 1.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        results.append({"threshold": t, "precision": precision, "recall": recall, "f1": f1,
                         "tp": tp, "fp": fp, "fn": fn, "tn": tn})
    return results


def print_table(results, title):
    print(f"\n=== {title} ===")
    print(f"{'thr':>6} {'prec':>6} {'rec':>6} {'f1':>6} {'tp':>3} {'fp':>3} {'fn':>3} {'tn':>3}")
    best = max(results, key=lambda r: r["f1"])
    for r in results:
        marker = "  <-- best F1" if r["threshold"] == best["threshold"] else ""
        print(f"{r['threshold']:6.3f} {r['precision']:6.3f} {r['recall']:6.3f} {r['f1']:6.3f} "
              f"{r['tp']:3d} {r['fp']:3d} {r['fn']:3d} {r['tn']:3d}{marker}")
    print(f"\nRecommended {title} threshold: {best['threshold']:.3f} "
          f"(F1={best['f1']:.3f}, precision={best['precision']:.3f}, recall={best['recall']:.3f})")
    return best["threshold"]


def main():
    print("Connecting to vector DB and loading models (embedding + reranker)...")
    client, collection_name = load_data()
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)
    reranker = Reranker()

    # Pull a sample of topics straight out of the live collection via a
    # cheap scroll, so the eval set matches what's actually indexed.
    sample_points, _ = client.scroll(collection_name=collection_name, limit=2000, with_payload=True)
    topics_seen = [p.payload.get("topic") for p in sample_points]

    records = build_eval_queries(topics_seen)
    print(f"Built {len(records)} weakly-labeled eval queries "
          f"({sum(r['should_answer'] for r in records)} positive / "
          f"{sum(not r['should_answer'] for r in records)} negative).\n")

    for i, r in enumerate(records, start=1):
        candidates = search(r["query"], model, client, collection_name, top_k=RETRIEVAL_TOP_K)
        reranked = reranker.rerank(r["query"], candidates, top_k=FINAL_TOP_K)
        r["score"] = reranked[0]["score"] if reranked else 0.0
        r["rerank_score"] = reranked[0]["rerank_score"] if reranked else -999.0
        print(f"  [{i}/{len(records)}] '{r['query'][:60]}' -> score={r['score']:.3f} "
              f"rerank={r['rerank_score']:.3f} (expected_answer={r['should_answer']})")

    score_values = sorted(r["score"] for r in records)
    rerank_values = sorted(r["rerank_score"] for r in records)
    score_thresholds = sorted(set(round(v, 3) for v in score_values))
    rerank_thresholds = sorted(set(round(v, 3) for v in rerank_values))

    best_score_thr = print_table(sweep(records, "score", score_thresholds), "raw cosine 'score'")
    best_rerank_thr = print_table(sweep(records, "rerank_score", rerank_thresholds), "'rerank_score'")

    print("\n=== ACTION ===")
    print(f"guardrails/confidence.py currently gates on 'rerank_score' first (falls back to 'score').")
    print(f"Set RETRIEVAL_CONFIDENCE_THRESHOLD={best_rerank_thr:.3f} in your .env "
          f"(measured on YOUR data, not a placeholder).")
    print(f"If you switch confidence.py to gate on the raw cosine score instead, "
          f"use {best_score_thr:.3f}.")


if __name__ == "__main__":
    main()


Connecting to vector DB and loading models (embedding + reranker)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Built 19 weakly-labeled eval queries (7 positive / 12 negative).



config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

...

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  [1/19] 'How is Allergy treated?' -> score=0.863 rerank=1.915 (expected_answer=True)
  [2/19] 'What causes Anxiety Disorders?' -> score=0.874 rerank=5.762 (expected_answer=True)
  [3/19] 'ما هي أعراض Acute Bronchitis؟' -> score=0.837 rerank=7.442 (expected_answer=True)
  [4/19] 'What are the symptoms of Anemia?' -> score=0.871 rerank=7.345 (expected_answer=True)
  [5/19] 'What causes The Gale Encyclopedia of Medicine (Second Editio' -> score=0.867 rerank=5.086 (expected_answer=True)
  [6/19] 'How is Alzheimer'S Disease treated?' -> score=0.859 rerank=7.557 (expected_answer=True)
  [7/19] 'ما هي أعراض Asthma؟' -> score=0.819 rerank=3.211 (expected_answer=True)
  [8/19] 'What is the best smartphone to buy this year?' -> score=0.751 rerank=-5.085 (expected_answer=False)
  [9/19] 'How do airplanes stay in the air?' -> score=0.823 rerank=-2.779 (expected_answer=False)
  [10/19] 'What is the capital of Japan?' -> score=0.778 rerank=-2.081 (expected_answer=False)
  [11/19] 'How do I bake a c

### `rag_pipeline.py`

In [24]:
# ==== rag_pipeline.py ====
"""
RAG pipeline for the medical assistant using Google GenAI SDK.

=== EDIT LOG (see CHANGELOG.md for the full walkthrough) ===
- [item #10] `answer()` now runs guardrails in this exact order:
  Emergency -> Vagueness -> Retrieval(top20) -> Rerank(top5) ->
  Confidence -> Gemini. Each guardrail can short-circuit and return
  immediately, before Gemini or even retrieval is touched.
- [item #18] Query rewriting is now CONDITIONAL (`_needs_rewrite`)
  instead of running on every single query - skips the extra LLM call
  when the query is already clear, specific, English medical language.
- [item #19] Retrieval now asks for RETRIEVAL_TOP_K=20 candidates and
  passes them through reranker.Reranker to cut down to FINAL_TOP_K=5.
- [item #14-17] SYSTEM_PROMPT_TEMPLATE rewritten: natural tone instead
  of "the provided excerpts do not mention...", natural end-of-paragraph
  citations instead of one citation per sentence, explicit instruction
  to say "I don't have enough information" rather than hedge/hallucinate
  when the evidence genuinely doesn't support an answer.
- [item #7] `retrieve()` now goes through the Qdrant-backed search.py
  (embedding_model + qdrant client) instead of loading local .npy files.
"""

import os
import time
import concurrent.futures
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# [Colab] All of these are already defined by cells you ran earlier in
# this same notebook (search.py, guardrails/emergency.py, guardrails/vagueness.py,
# guardrails/confidence.py, reranker.py) - there are no real files/packages on
# disk to import from here, so don't try to `import` them; just make sure
# those cells ran (in order, top to bottom) before this one.

# تحميل متغيرات البيئة تلقائياً من ملف .env
load_dotenv()

# ===== Configuration =====
EMBEDDING_MODEL_NAME = "intfloat/multilingual-e5-base"
LLM_MODEL_NAME = "gemini-3.6-flash"
LLM_MODEL_CHAIN = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
]
PER_CALL_TIMEOUT_SECONDS = 8
MAX_TOTAL_LLM_SECONDS = 15
MAX_CONTEXT_TOKENS = 3500
LLM_MAX_TOKENS = 1024
RETRIEVAL_TOP_K = 20   # [item #19] wide candidate pool for the reranker
FINAL_TOP_K = 5        # [item #19] what actually goes into the prompt

FALLBACK_MESSAGE = "هذه المعلومات غير متوفرة في المصادر الحالية. يرجى استشارة طبيب متخصص."

# [item #14-17] Rewritten to sound natural and to give a clear, non-hedgy
# instruction for the "not enough evidence" case instead of "the provided
# excerpts do not mention...".
SYSTEM_PROMPT_TEMPLATE = """You are a medical information assistant speaking directly to the user. \
Answer in the same language as the question (Arabic question -> Arabic answer, English -> English), \
and answer naturally and directly, the way a knowledgeable person would - never mention "the provided \
excerpts", "the retrieved context", "the sources above", vector search, or any detail of how you found \
the information.

Use the retrieved sources below as your factual evidence for this answer. Treat them as authoritative:
- Do not introduce medical claims that are not supported by them.
- Do not give precise medication dosages even if mentioned in the text - direct the user to a doctor or pharmacist for that.
- Do not state a definitive diagnosis. If the evidence mentions matching conditions, present them as general possibilities discussed in medical literature (e.g. "this can be seen with asthma or bronchitis"), and note that a specialist can confirm.
- Add a source citation naturally at the end of a relevant paragraph (not after every sentence), e.g. "... (Wikipedia - Asthma)".
- If the evidence genuinely does not support an answer to this specific question, say plainly that you don't have enough reliable information to answer that, and suggest what extra detail (duration, other symptoms, severity) would help - do NOT say the excerpts/sources "don't mention" it, just say you don't have enough information.
- End every response with a brief reminder that this is not a substitute for seeing a qualified doctor.

Retrieved evidence:
{context}

User question: {query}

Answer:"""


def _needs_rewrite(query: str) -> bool:
    """[item #18] Skip the extra LLM call when the query is already
    clear enough for vector search: pure-English queries of a decent
    length are assumed to already use reasonable medical/search terms.
    Short queries and anything containing Arabic (colloquial symptom
    descriptions) still get rewritten/translated."""
    import re
    has_arabic = bool(re.search(r"[\u0600-\u06FF]", query))
    if has_arabic:
        return True
    return len(query.split()) < 4


class RAGPipeline:
    """Encapsulates guardrails, retrieval, reranking, prompt construction,
    and LLM calls for the medical assistant."""

    def __init__(
        self,
        embedding_model_name: str = EMBEDDING_MODEL_NAME,
        llm_model_name: str = LLM_MODEL_NAME,
        max_context_tokens: int = MAX_CONTEXT_TOKENS,
        llm_max_tokens: int = LLM_MAX_TOKENS,
    ):
        self.max_context_tokens = max_context_tokens
        self.llm_max_tokens = llm_max_tokens
        self.llm_model_name = llm_model_name

        print("Connecting to vector DB and loading embedding model...")
        self.qdrant_client, self.collection_name = load_data()
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.reranker = Reranker()  # lazy-loads its own model on first use
        print("Loaded successfully.")

        self._llm_client = None

    @property
    def llm_client(self):
        """Lazily initialize the Gemini client so import time stays fast."""
        if self._llm_client is None:
            from google import genai
            self._llm_client = genai.Client()
        return self._llm_client

    @staticmethod
    def estimate_tokens(text: str) -> int:
        """Rough token count estimate based on whitespace-separated words."""
        return int(len(text.split()) * 1.3)

    def rewrite_query(self, query: str) -> str:
        """Translates and expands colloquial user queries into standard
        English medical terminology to optimize vector search cosine
        similarity. [item #18] Only called when _needs_rewrite() is True."""
        rewrite_prompt = f"""You are a medical search query generator.
Convert the following user symptom complaint or question into concise, standard English medical terminology keywords suitable for searching a medical textbook index or vector database.
Keep it strictly to key clinical terms/symptoms without conversational filler.

User Input: {query}
Search Keywords (in English):"""
        try:
            expanded_query = self.call_llm(rewrite_prompt)
            print(f"\n[Query Rewritten for Search]: {expanded_query.strip()}")
            return expanded_query.strip()
        except Exception:
            return query  # Fallback to original query on error

    def retrieve(self, query: str, top_k: int = RETRIEVAL_TOP_K):
        """Runs search() against the Qdrant collection to get the raw
        top-k candidates (before reranking)."""
        return search(
            query,
            self.embedding_model,
            self.qdrant_client,
            self.collection_name,
            top_k=top_k,
        )

    def build_context(self, search_query: str, reranked_results: list):
        """Trims the already-reranked top results to stay within the
        max token budget. Returns (context_text, sources_list)."""
        context_parts = []
        sources = []
        total_tokens = 0

        for result in reranked_results:
            chunk_text = result.get("text", "")
            chunk_tokens = self.estimate_tokens(chunk_text)

            if total_tokens + chunk_tokens > self.max_context_tokens:
                break

            source_name = result.get("source_file") or result.get("topic") or "Unknown Source"

            context_parts.append(f"[Source: {source_name}]\n{chunk_text}")
            sources.append({
                "source": source_name,
                "score": round(float(result.get("score", 0.0)), 3),
                "rerank_score": round(float(result.get("rerank_score", 0.0)), 3),
            })
            total_tokens += chunk_tokens

        context_text = "\n\n---\n\n".join(context_parts)
        return context_text, sources

    def build_prompt(self, query: str, context_text: str) -> str:
        return SYSTEM_PROMPT_TEMPLATE.format(context=context_text, query=query)

    def _call_one_model(self, model_name: str, prompt: str) -> str:
        """نداء واحد لموديل واحد، بحد أقصى زمني صارم عن طريق thread."""
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
            future = executor.submit(
                lambda: self.llm_client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                ).text
            )
            return future.result(timeout=PER_CALL_TIMEOUT_SECONDS)

    def call_llm(self, prompt: str) -> str:
        """
        يجرب كل موديل في LLM_MODEL_CHAIN بالترتيب.
        - لو 503/UNAVAILABLE أو timeout: يروح للموديل اللي بعده فورًا (مفيش sleep طويل).
        - لو كل الموديلات فشلت أو الوقت الكلي عدّى MAX_TOTAL_LLM_SECONDS:
          يرجع None بدل ما يعمل raise ويكسّر الـ request كله.
        """
        start = time.monotonic()

        for model_name in LLM_MODEL_CHAIN:
            if time.monotonic() - start > MAX_TOTAL_LLM_SECONDS:
                break
            try:
                return self._call_one_model(model_name, prompt)
            except concurrent.futures.TimeoutError:
                print(f"\n[الموديل {model_name} بطيء جدًا، بجرب اللي بعده]")
                continue
            except Exception as e:
                err_str = str(e)
                if "503" in err_str or "UNAVAILABLE" in err_str:
                    print(f"\n[الموديل {model_name} مضغوط، بجرب اللي بعده]")
                    continue
                raise

        return None

    def answer(self, query: str) -> dict:
        """
        Full pipeline, guardrails-first (item #10):
          1. Emergency guardrail (deterministic, no LLM/retrieval)
          2. Vagueness guardrail (rule-based, LLM fallback only if ambiguous)
          3. Retrieval (top-20) + reranking (top-5)
          4. Confidence guardrail on the reranked top score
          5. Prompt + Gemini call
        """
        # 1. Emergency
        emergency_category = check_emergency(query)
        if emergency_category:
            return {
                "answer": get_emergency_response(emergency_category),
                "sources": [],
                "emergency": True,
                "guardrail": None,
            }

        # 2. Vagueness (rule-based first; LLM fallback left available for
        # you to wire in for genuinely ambiguous middle-length queries -
        # see guardrails/vagueness.llm_vagueness_check)
        vague_check = is_vague(query)
        if not vague_check["allowed"]:
            return {
                "answer": vague_check["message"],
                "sources": [],
                "emergency": False,
                "guardrail": vague_check["reason"],
            }

        # 3. Retrieval + rerank
        search_query = self.rewrite_query(query) if _needs_rewrite(query) else query
        candidates = self.retrieve(search_query, top_k=RETRIEVAL_TOP_K)
        reranked = self.reranker.rerank(search_query, candidates, top_k=FINAL_TOP_K)

        # 4. Confidence guardrail
        confidence_check = check_confidence(reranked, query=query)
        if not confidence_check["allowed"]:
            return {
                "answer": confidence_check["message"],
                "sources": [],
                "emergency": False,
                "guardrail": confidence_check["reason"],
            }

        context_text, sources = self.build_context(search_query, reranked)
        if not context_text.strip():
            return {"answer": FALLBACK_MESSAGE, "sources": [], "emergency": False, "guardrail": "no_context"}

        # 5. Generation
        prompt = self.build_prompt(query, context_text)
        answer_text = self.call_llm(prompt)

        if answer_text is None:
            return {
                "answer": "الخدمة مشغولة جدًا دلوقتي، جرب تاني بعد شوية. لو الأعراض مقلقة، متأخرش في استشارة طبيب.",
                "sources": [],
                "emergency": False,
                "guardrail": "llm_unavailable",
            }

        return {"answer": answer_text, "sources": sources, "emergency": False, "guardrail": None}


def run_interactive_session(pipeline: RAGPipeline) -> None:
    """Simple REPL loop for testing the pipeline from the terminal."""
    print("Type your question (or 'exit' to quit):")

    while True:
        query = input("\nQuestion: ").strip()

        if query.lower() in ("exit", "quit"):
            break
        if not query:
            continue

        result = pipeline.answer(query)

        print("\n=== Answer ===")
        print(result["answer"])
        print("\n=== Sources ===")
        print(result["sources"])


if __name__ == "__main__":
    rag_pipeline = RAGPipeline()
    run_interactive_session(rag_pipeline)


Connecting to vector DB and loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loaded successfully.
Type your question (or 'exit' to quit):

Question: عندي سكر من سنتين وحاسس بعطش زيادة وتبول كتير، هل ده طبيعي؟


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


=== Answer ===
معنديش معلومات كافية وموثوقة في المصادر المتاحة عشان أجاوبك بدقة على السؤال ده. لو تقدر تحدد الأعراض أو الموضوع أكتر، هحاول أساعدك تاني، أو يفضل تستشير طبيب مباشرة.

=== Sources ===
[]

Question: ما هي أعراض الغدة الدرقية الخاملة (قصور الغدة الدرقية)؟


ValueError: No API key was provided. Please pass a valid API key. Learn how to create an API key at https://ai.google.dev/gemini-api/docs/api-key.

### `main.py`

In [23]:
# ==== main.py ====
"""
[NEW FILE - items #25-27 of the plan]

FastAPI wrapper around RAGPipeline. This is the ONLY thing that should
run in production as a live server; everything upstream (extraction,
chunking, embeddings, upload_to_vector_db.py) is an offline, one-time-ish
pipeline you run yourself before deploying this.

Run locally:
    uvicorn main:app --reload --port 8000

Then:
    curl -X POST http://localhost:8000/chat -H "Content-Type: application/json" \
         -d '{"question": "عندي كحة من أسبوع ومعاها حرارة"}'
"""

from contextlib import asynccontextmanager

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

# [Colab] RAGPipeline is already defined by the rag_pipeline.py cell above -
# no real rag_pipeline.py file exists here to import from.
# NOTE: main.py is meant to become a REAL file you deploy with
# `uvicorn main:app`, which needs actual files on disk - this notebook cell
# lets you define/test the FastAPI `app` object inline, but running it as a
# live server is a separate, later step (see DEPLOYMENT.md at the bottom).

pipeline_holder = {}


@asynccontextmanager
async def lifespan(app: FastAPI):
    # Loaded once at startup - NOT per-request - so the embedding model
    # and reranker are only initialized a single time.
    print("Starting up: loading RAG pipeline...")
    pipeline_holder["pipeline"] = RAGPipeline()
    yield
    pipeline_holder.clear()


app = FastAPI(title="Medical RAG Assistant", lifespan=lifespan)

# Tighten this to your actual frontend domain(s) before going to production.
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class ChatRequest(BaseModel):
    question: str


class ChatResponse(BaseModel):
    answer: str
    sources: list
    emergency: bool
    guardrail: str | None = None


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/chat", response_model=ChatResponse)
def chat(request: ChatRequest):
    pipeline: RAGPipeline = pipeline_holder["pipeline"]
    result = pipeline.answer(request.question)
    return ChatResponse(**result)


## DEPLOYMENT.md

# دليل الديبلويمنت (خطوة بخطوة)

## المرحلة 1: الـ Offline pipeline (تشتغل مرة، عندك أو على Colab)
```
python build_dataset_from_medquad.py
python build_dataset_from_wikipedia_category.py
python extract_book_pdf.py
python process_documents.py
python generate_embeddings.py
```

## المرحلة 2: Vector DB خارجي (Qdrant Cloud - فيه free tier)
1. اعمل حساب على https://cloud.qdrant.io وأنشئ cluster مجاني.
2. هتاخد `QDRANT_URL` و `QDRANT_API_KEY` من لوحة التحكم.
3. اعمل نسخة من `.env.example` باسم `.env` واملأ القيم.
4. `pip install qdrant-client python-dotenv`
5. `python upload_to_vector_db.py`

## المرحلة 3: تجربة محلية للـ API
```
pip install -r requirements.txt
uvicorn main:app --reload --port 8000
curl -X POST http://localhost:8000/chat -H "Content-Type: application/json" -d '{"question":"ما هي أعراض الربو؟"}'
```

## المرحلة 4: الديبلويمنت الفعلي
اختار أي منصة بتستضيف حاويات/Python apps وعندها free/cheap tier، زي Render أو Railway أو Fly.io:
1. اربط الـ repo بتاعك بالمنصة.
2. Start command: `uvicorn main:app --host 0.0.0.0 --port $PORT`
3. حط الـ environment variables (`QDRANT_URL`, `QDRANT_API_KEY`, `QDRANT_COLLECTION`, `GOOGLE_API_KEY`, `RETRIEVAL_CONFIDENCE_THRESHOLD`) في إعدادات المنصة - **متحطش الـ `.env` في الـ repo نفسه**.
4. الداتا (`chunks.json`, `embeddings.npy`) مش محتاجة تتنشر مع الـ API خالص - هي بس بتتستخدم مرة واحدة وقت `upload_to_vector_db.py`. الـ API server نفسه محتاج بس الكود + الـ requirements.

## ملحوظات مهمة
- الـ preprocessing pipeline (المرحلة 1) منفصلة تمامًا عن الـ production server. متشغلش استخراج البيانات جوه FastAPI أو جوه request handler.
- قبل ما تعتمد على `RETRIEVAL_CONFIDENCE_THRESHOLD` الافتراضي، اعمل تقييم بسيط زي ما موضح في `guardrails/confidence.py`.
- CORS في `main.py` مفتوح لأي origin (`*`) للتجربة بس - ضيّقه لدومين الـ frontend بتاعك قبل الإطلاق الفعلي.
